# PERSUADE: Local Lag Curve Analysis

**Goal:** Measure short-range context sensitivity by varying how much immediately preceding context the model gets.

**Hypothesis:** Group differences (quality / grade / ELL) may show up as different shapes in the short-lag dependency curve (how quickly influence saturates within the most recent 5-100 tokens).

**Method:**
- For each essay, predict a fixed scoring region (last 256 tokens)
- Vary context length k from 0 to 128 tokens immediately preceding the target
- Compute NLL(k) and gain_k = NLL(0) - NLL(k)
- Compare curve shapes across groups

**Cohort:** persuade_score_long_cohort.jsonl (50/50/50 balanced, all long enough)

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from scipy.optimize import curve_fit
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'local_lag_curve_v1'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = True  # True for T4, False for A100

# Scoring region: last 256 tokens of each essay
TARGET_LENGTH = 256

# Context length grid (k values)
K_GRID = [0, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128]

# Minimum tokens required: context pool (128) + target (256)
MIN_TOKENS_REQUIRED = max(K_GRID) + TARGET_LENGTH

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nTarget length: {TARGET_LENGTH} tokens (last 256 of each essay)")
print(f"Context grid k: {K_GRID}")
print(f"Min tokens required: {MIN_TOKENS_REQUIRED}")

## 1. Load Model and Data

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Load cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
print(f"Loaded {len(cohort)} essays")

# Verify cohort
df_cohort = pd.DataFrame(cohort)
print(f"\nScore bin distribution:")
print(df_cohort['score_bin'].value_counts())

In [ ]:
# Pre-tokenize and filter
essay_tokens = {}
excluded = 0

for essay in cohort:
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    if len(token_ids) >= MIN_TOKENS_REQUIRED:
        essay_tokens[essay['essay_id']] = token_ids
    else:
        excluded += 1

print(f"Tokenized essays: {len(essay_tokens)}")
print(f"Excluded (too short): {excluded}")

lengths = [len(t) for t in essay_tokens.values()]
print(f"Token lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")

## 2. Core Functions

In [ ]:
@torch.no_grad()
def compute_nll_for_target(context_ids, target_ids):
    """
    Compute mean NLL for predicting target_ids given context_ids.
    
    Args:
        context_ids: list of token ids for context (can be empty)
        target_ids: list of token ids for target region
    
    Returns:
        mean NLL over target tokens
    """
    # Combine context + target
    full_ids = context_ids + target_ids
    
    # If no context, we still need at least the first target token as "prompt"
    if len(context_ids) == 0:
        # We'll predict from position 0 onward (predicting token 1 given token 0, etc.)
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]  # shape: (seq_len, vocab_size)
        
        # Compute NLL for positions 0 to len-2 predicting tokens 1 to len-1
        nlls = []
        for i in range(len(full_ids) - 1):
            log_probs = torch.log_softmax(logits[i], dim=-1)
            true_token = full_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        return np.mean(nlls) if nlls else 0.0
    else:
        # With context: predict target tokens given context
        input_ids = torch.tensor([full_ids], device=model.device)
        outputs = model(input_ids)
        logits = outputs.logits[0]
        
        # Compute NLL only for target region
        # Position i in logits predicts token i+1
        # Target starts at position len(context_ids)
        # So we use logits[len(context_ids)-1] to predict target[0], etc.
        target_start = len(context_ids)
        nlls = []
        
        for i in range(len(target_ids) - 1):
            pos = target_start + i  # position in full sequence
            if pos >= logits.shape[0]:
                break
            log_probs = torch.log_softmax(logits[pos], dim=-1)
            true_token = target_ids[i + 1]
            nll = -log_probs[true_token].item()
            nlls.append(nll)
        
        # Also include prediction of first target token from last context token
        if len(context_ids) > 0:
            log_probs = torch.log_softmax(logits[len(context_ids) - 1], dim=-1)
            nll_first = -log_probs[target_ids[0]].item()
            nlls.insert(0, nll_first)
        
        return np.mean(nlls) if nlls else 0.0


def run_lag_curve_for_essay(token_ids, essay_id):
    """
    Compute NLL for each k in K_GRID.
    
    Returns dict with k -> nll mapping and derived metrics.
    """
    n_tokens = len(token_ids)
    
    # Target = last TARGET_LENGTH tokens
    target_start = n_tokens - TARGET_LENGTH
    target_ids = token_ids[target_start:]
    
    # Context pool = everything before target
    context_pool = token_ids[:target_start]
    
    results = {'essay_id': essay_id}
    nll_by_k = {}
    
    for k in K_GRID:
        if k == 0:
            # No context - just target tokens
            context_ids = []
        else:
            # Last k tokens of context pool
            context_ids = context_pool[-k:]
        
        nll = compute_nll_for_target(context_ids, target_ids)
        nll_by_k[k] = nll
        results[f'nll_{k}'] = nll
    
    # Compute gains (improvement from k=0)
    nll_0 = nll_by_k[0]
    for k in K_GRID:
        gain_k = nll_0 - nll_by_k[k]
        results[f'gain_{k}'] = gain_k
    
    # Shape metrics
    gain_16 = results['gain_16']
    gain_128 = results['gain_128']
    
    # early_ratio: fraction of benefit achieved by k=16
    if gain_128 > 0:
        results['early_ratio'] = gain_16 / gain_128
    else:
        results['early_ratio'] = np.nan
    
    # log_slope_local: fit line to (log(k+1), gain_k) for k >= 2
    ks_for_fit = [k for k in K_GRID if k >= 2]
    log_ks = [np.log(k + 1) for k in ks_for_fit]
    gains_for_fit = [results[f'gain_{k}'] for k in ks_for_fit]
    
    if len(log_ks) >= 2:
        slope, intercept, r_value, p_value, std_err = stats.linregress(log_ks, gains_for_fit)
        results['log_slope_local'] = slope
        results['log_slope_r2'] = r_value ** 2
    else:
        results['log_slope_local'] = np.nan
        results['log_slope_r2'] = np.nan
    
    # AUC under gain curve (normalized by max possible)
    # Use trapezoidal rule on log scale
    auc = 0
    for i in range(len(K_GRID) - 1):
        k1, k2 = K_GRID[i], K_GRID[i + 1]
        g1, g2 = results[f'gain_{k1}'], results[f'gain_{k2}']
        # Width on log scale
        width = np.log(k2 + 1) - np.log(k1 + 1)
        auc += 0.5 * (g1 + g2) * width
    results['auc_local'] = auc
    
    # Baseline NLL (with full context k=128)
    results['baseline_nll'] = nll_by_k[128]
    
    return results


print("Core functions defined")

## 3. Run Lag Curve Analysis

In [ ]:
# Build essay metadata lookup
essay_meta = {e['essay_id']: e for e in cohort}

# Process all essays
all_results = []
start_time = time.time()

for essay_id, token_ids in tqdm(essay_tokens.items(), desc="Processing essays"):
    results = run_lag_curve_for_essay(token_ids, essay_id)
    
    # Add metadata
    meta = essay_meta[essay_id]
    results['score'] = meta.get('score')
    results['score_bin'] = meta.get('score_bin')
    results['grade'] = meta.get('grade')
    results['token_count'] = len(token_ids)
    
    all_results.append(results)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(essay_tokens):.2f}s/essay)")
print(f"Total essays: {len(all_results)}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_results)

print(f"Results shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")

df.head()

## 4. Sanity Checks

In [ ]:
print("="*80)
print("SANITY CHECK: NLL and Gain Distributions")
print("="*80)

print("\nNLL by k (should decrease as k increases):")
for k in K_GRID:
    col = f'nll_{k}'
    print(f"  k={k:3d}: mean={df[col].mean():.4f}, std={df[col].std():.4f}")

print("\nGain by k (should increase as k increases):")
for k in K_GRID:
    col = f'gain_{k}'
    print(f"  k={k:3d}: mean={df[col].mean():.4f}, std={df[col].std():.4f}")

In [ ]:
print("\n" + "="*80)
print("SANITY CHECK: Shape Metrics")
print("="*80)

print(f"\nearly_ratio (gain_16 / gain_128):")
print(f"  mean={df['early_ratio'].mean():.4f}, median={df['early_ratio'].median():.4f}")
print(f"  range=[{df['early_ratio'].min():.4f}, {df['early_ratio'].max():.4f}]")
print(f"  NaN count: {df['early_ratio'].isna().sum()}")

print(f"\nlog_slope_local:")
print(f"  mean={df['log_slope_local'].mean():.4f}, median={df['log_slope_local'].median():.4f}")
print(f"  range=[{df['log_slope_local'].min():.4f}, {df['log_slope_local'].max():.4f}]")

print(f"\ngain_128 (total local benefit):")
print(f"  mean={df['gain_128'].mean():.4f}, median={df['gain_128'].median():.4f}")
print(f"  range=[{df['gain_128'].min():.4f}, {df['gain_128'].max():.4f}]")

## 5. Group Analysis

In [ ]:
GROUP_ORDER = ['low', 'mid', 'high']
SCORE_COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}

df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)

In [ ]:
print("="*80)
print("GROUP MEANS: Shape Metrics by Score Bin")
print("="*80)

for metric in ['early_ratio', 'log_slope_local', 'gain_128', 'baseline_nll']:
    print(f"\n{metric}:")
    for score_bin in GROUP_ORDER:
        subset = df[df['score_bin'] == score_bin][metric].dropna()
        mean = subset.mean()
        sem = subset.sem()
        print(f"  {score_bin}: {mean:.4f} +/- {1.96*sem:.4f} (n={len(subset)})")

In [ ]:
# Build gain curve summary by group
curve_rows = []

for score_bin in GROUP_ORDER:
    subset = df[df['score_bin'] == score_bin]
    for k in K_GRID:
        gain_col = f'gain_{k}'
        nll_col = f'nll_{k}'
        
        curve_rows.append({
            'score_bin': score_bin,
            'k': k,
            'gain_mean': subset[gain_col].mean(),
            'gain_sem': subset[gain_col].sem(),
            'gain_ci95': 1.96 * subset[gain_col].sem(),
            'nll_mean': subset[nll_col].mean(),
            'nll_sem': subset[nll_col].sem(),
            'n': len(subset),
        })

df_curve = pd.DataFrame(curve_rows)
print("Group curve summary:")
print(df_curve.pivot(index='k', columns='score_bin', values='gain_mean').round(4))

## 6. Statistical Tests

In [ ]:
# Standardize controls
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()
df['baseline_nll_z'] = (df['baseline_nll'] - df['baseline_nll'].mean()) / df['baseline_nll'].std()

# Drop rows with NaN in key metrics
df_valid = df.dropna(subset=['early_ratio', 'log_slope_local']).copy()
print(f"Valid essays for regression: {len(df_valid)} / {len(df)}")

In [ ]:
print("="*80)
print("REGRESSION: early_ratio ~ C(score_bin) + controls")
print("="*80)

formula_early = 'early_ratio ~ C(score_bin) + token_count_z + baseline_nll_z'
model_early = smf.ols(formula_early, data=df_valid).fit()

print(f"\nFormula: {formula_early}")
print(f"R²: {model_early.rsquared:.4f}, Adj R²: {model_early.rsquared_adj:.4f}, n={int(model_early.nobs)}")

print("\nCoefficients:")
for param in model_early.params.index:
    coef = model_early.params[param]
    pval = model_early.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
print("\n" + "="*80)
print("REGRESSION: log_slope_local ~ C(score_bin) + controls")
print("="*80)

formula_slope = 'log_slope_local ~ C(score_bin) + token_count_z + baseline_nll_z'
model_slope = smf.ols(formula_slope, data=df_valid).fit()

print(f"\nFormula: {formula_slope}")
print(f"R²: {model_slope.rsquared:.4f}, Adj R²: {model_slope.rsquared_adj:.4f}, n={int(model_slope.nobs)}")

print("\nCoefficients:")
for param in model_slope.params.index:
    coef = model_slope.params[param]
    pval = model_slope.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
print("\n" + "="*80)
print("REGRESSION: gain_128 ~ C(score_bin) + controls")
print("="*80)

formula_gain = 'gain_128 ~ C(score_bin) + token_count_z + baseline_nll_z'
model_gain = smf.ols(formula_gain, data=df_valid).fit()

print(f"\nFormula: {formula_gain}")
print(f"R²: {model_gain.rsquared:.4f}, Adj R²: {model_gain.rsquared_adj:.4f}, n={int(model_gain.nobs)}")

print("\nCoefficients:")
for param in model_gain.params.index:
    coef = model_gain.params[param]
    pval = model_gain.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

## 7. Visualizations

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_BASE) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

In [ ]:
# PLOT 1: Group mean gain curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Linear x-axis
ax = axes[0]
for score_bin in GROUP_ORDER:
    subset = df_curve[df_curve['score_bin'] == score_bin]
    ax.errorbar(subset['k'], subset['gain_mean'], yerr=subset['gain_ci95'],
                marker='o', capsize=3, label=score_bin, color=SCORE_COLORS[score_bin],
                linewidth=2, markersize=6)

ax.set_xlabel('Context length k (tokens)', fontsize=11)
ax.set_ylabel('Gain = NLL(0) - NLL(k)', fontsize=11)
ax.set_title('Gain Curve by Score Bin\n(linear scale)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

# Right: Log x-axis
ax = axes[1]
for score_bin in GROUP_ORDER:
    subset = df_curve[df_curve['score_bin'] == score_bin]
    # Skip k=0 for log scale
    subset_nonzero = subset[subset['k'] > 0]
    ax.errorbar(subset_nonzero['k'], subset_nonzero['gain_mean'], yerr=subset_nonzero['gain_ci95'],
                marker='o', capsize=3, label=score_bin, color=SCORE_COLORS[score_bin],
                linewidth=2, markersize=6)

ax.set_xscale('log')
ax.set_xlabel('Context length k (tokens, log scale)', fontsize=11)
ax.set_ylabel('Gain = NLL(0) - NLL(k)', fontsize=11)
ax.set_title('Gain Curve by Score Bin\n(log scale)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(output_dir / 'gain_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT 2: Distribution of early_ratio by group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
ax = axes[0]
data_for_plot = [df[df['score_bin'] == sb]['early_ratio'].dropna() for sb in GROUP_ORDER]
parts = ax.violinplot(data_for_plot, positions=range(len(GROUP_ORDER)), showmeans=True, showmedians=True)

for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(SCORE_COLORS[GROUP_ORDER[i]])
    pc.set_alpha(0.7)

ax.set_xticks(range(len(GROUP_ORDER)))
ax.set_xticklabels(GROUP_ORDER)
ax.set_xlabel('Score Bin', fontsize=11)
ax.set_ylabel('early_ratio (gain_16 / gain_128)', fontsize=11)
ax.set_title('Distribution of Early Saturation Ratio\n(higher = faster saturation)', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Box plot
ax = axes[1]
df_valid.boxplot(column='early_ratio', by='score_bin', ax=ax,
                  positions=range(len(GROUP_ORDER)))
ax.set_xlabel('Score Bin', fontsize=11)
ax.set_ylabel('early_ratio', fontsize=11)
ax.set_title('early_ratio by Score Bin', fontsize=12, fontweight='bold')
plt.suptitle('')  # Remove automatic title
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(output_dir / 'early_ratio_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT 3: Scatter early_ratio vs baseline_nll
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# early_ratio vs baseline_nll
ax = axes[0]
for score_bin in GROUP_ORDER:
    subset = df_valid[df_valid['score_bin'] == score_bin]
    ax.scatter(subset['baseline_nll'], subset['early_ratio'], 
               alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], s=30)

ax.set_xlabel('Baseline NLL (fluency proxy)', fontsize=11)
ax.set_ylabel('early_ratio', fontsize=11)
ax.set_title('early_ratio vs Fluency\n(checking independence)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

# Correlation
r, p = stats.pearsonr(df_valid['baseline_nll'], df_valid['early_ratio'])
ax.text(0.05, 0.95, f'r = {r:.3f}, p = {p:.4f}', transform=ax.transAxes, 
        fontsize=10, verticalalignment='top')

# log_slope vs baseline_nll
ax = axes[1]
for score_bin in GROUP_ORDER:
    subset = df_valid[df_valid['score_bin'] == score_bin]
    ax.scatter(subset['baseline_nll'], subset['log_slope_local'], 
               alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], s=30)

ax.set_xlabel('Baseline NLL (fluency proxy)', fontsize=11)
ax.set_ylabel('log_slope_local', fontsize=11)
ax.set_title('Log Slope vs Fluency\n(checking independence)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

r, p = stats.pearsonr(df_valid['baseline_nll'], df_valid['log_slope_local'])
ax.text(0.05, 0.95, f'r = {r:.3f}, p = {p:.4f}', transform=ax.transAxes, 
        fontsize=10, verticalalignment='top')

plt.tight_layout()
plt.savefig(output_dir / 'shape_vs_fluency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PLOT 4: log_slope_local distribution
fig, ax = plt.subplots(figsize=(10, 5))

for score_bin in GROUP_ORDER:
    subset = df_valid[df_valid['score_bin'] == score_bin]['log_slope_local']
    ax.hist(subset, bins=25, alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], density=True)

ax.set_xlabel('log_slope_local', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of Log Slope (Saturation Rate)\n(higher = steeper saturation)', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'log_slope_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Results

In [ ]:
# Save essay-level results
df.to_csv(output_dir / 'essay_level_results_local_lag.csv', index=False)

# Save curve data in long format
curve_long_rows = []
for _, row in df.iterrows():
    for k in K_GRID:
        curve_long_rows.append({
            'essay_id': row['essay_id'],
            'score_bin': row['score_bin'],
            'k': k,
            'nll_k': row[f'nll_{k}'],
            'gain_k': row[f'gain_{k}'],
        })

df_curve_long = pd.DataFrame(curve_long_rows)
df_curve_long.to_csv(output_dir / 'curve_long.csv', index=False)

# Save group curve summary
df_curve.to_csv(output_dir / 'group_curve_summary.csv', index=False)

# Save regression results
with open(output_dir / 'regression_local_lag.txt', 'w') as f:
    f.write("LOCAL LAG CURVE REGRESSION ANALYSIS\n")
    f.write("=" * 80 + "\n\n")
    f.write(f"Configuration:\n")
    f.write(f"  Cohort: persuade_score_long (50/50/50)\n")
    f.write(f"  Target length: {TARGET_LENGTH} tokens\n")
    f.write(f"  Context grid k: {K_GRID}\n")
    f.write(f"  Essays analyzed: {len(df)}\n")
    f.write("\n" + "=" * 80 + "\n\n")
    f.write("EARLY_RATIO MODEL:\n")
    f.write(f"Formula: {formula_early}\n")
    f.write(model_early.summary().as_text())
    f.write("\n\n" + "=" * 80 + "\n\n")
    f.write("LOG_SLOPE_LOCAL MODEL:\n")
    f.write(f"Formula: {formula_slope}\n")
    f.write(model_slope.summary().as_text())
    f.write("\n\n" + "=" * 80 + "\n\n")
    f.write("GAIN_128 MODEL:\n")
    f.write(f"Formula: {formula_gain}\n")
    f.write(model_gain.summary().as_text())

print(f"\nSaved to {output_dir}/")
print(f"  - essay_level_results_local_lag.csv ({len(df)} rows)")
print(f"  - curve_long.csv ({len(df_curve_long)} rows)")
print(f"  - group_curve_summary.csv")
print(f"  - regression_local_lag.txt")
print(f"  - gain_curves.png")
print(f"  - early_ratio_distribution.png")
print(f"  - shape_vs_fluency.png")
print(f"  - log_slope_distribution.png")

In [ ]:
# Final summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("""
GOAL: Measure short-range context sensitivity via prefix-length lag curves.

METHOD:
  - For each essay, predict last 256 tokens with varying context (k=0 to 128)
  - Compute gain_k = NLL(0) - NLL(k): improvement from adding k tokens of context
  - Extract shape metrics:
    * early_ratio = gain_16 / gain_128 (fraction of benefit from first 16 tokens)
    * log_slope_local = slope of gain vs log(k) (saturation rate)

KEY QUESTIONS:
  1. Do groups differ in curve shape (not just fluency)?
  2. Is early_ratio different across score bins?
  3. Is log_slope_local different across score bins?
  4. Are shape metrics independent of baseline fluency?

INTERPRETATION:
  - Higher early_ratio = faster saturation (first 16 tokens capture most benefit)
  - Higher log_slope_local = steeper gain curve
  - If groups differ in shape controlling for fluency, this suggests
    different dependency structures in the text itself.
""")